### 2. For this problem, we'll be using the animals with attributes data set and can find information about it.

In [ ]:
import numpy as np

# Load the 50 x 85 continuous feature matrix
X = np.loadtxt("predicate-matrix-continuous.txt")

# Load animal names
with open("classes.txt", "r") as f:
    animals = [line.strip().split("\t")[-1] for line in f]

print("X shape:", X.shape)
print("Number of animals:", len(animals))
print("First 5 animals:", animals[:5])

### (a) Ward hierarchical clustering + dendrogram

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

# Hierarchical clustering using Ward's method
R = linkage(X, method='ward')

# Set figure size
plt.figure(figsize=(10, 15))

# Plot the dendrogram
dendrogram(
    R,
    orientation='right',
    labels=animals,
    leaf_font_size=8
)

plt.title("Hierarchical Clustering of Animals (Ward's Method)")
plt.xlabel("Distance")
plt.ylabel("Animals")

plt.savefig("animal_dendrogram.png", dpi=200, bbox_inches="tight")
plt.show()

###(b)Extract the 10 clusters

In [ ]:

from scipy.cluster.hierarchy import fcluster

# Cut Ward's hierarchy into 10 clusters
clusters = fcluster(R, t=10, criterion='maxclust')

print(clusters)

### Show which animals belong to each cluster

In [ ]:
for i in range(1, 11):
    members = [animals[j] for j in range(len(animals)) if clusters[j] == i]
    print("Cluster", i, ":", members)

### Calculate Ward's k-means cost

In [ ]:
ward_cost = 0

for i in range(1, 11):
    points = X[clusters == i]
    center = np.mean(points, axis=0)
    ward_cost += np.sum((points - center) ** 2)

print("Ward k-means cost:", ward_cost)

### Run k-means 10 times

In [ ]:
from sklearn.cluster import KMeans

best_cost = float('inf')
best_labels = None
best_centers = None

for run in range(10):
    # Randomly choose 10 data points as initial centers
    indices = np.random.choice(len(X), 10, replace=False)
    initial_centers = X[indices]

    # Run k-means using these initial centers
    model = KMeans(
        n_clusters=10,
        init=initial_centers,
        n_init=1,
        random_state=run
    )

    model.fit(X)

    # k-means cost
    cost = model.inertia_

    print("Run", run + 1, "cost:", cost)

    # Keep the best solution
    if cost < best_cost:
        best_cost = cost
        best_labels = model.labels_
        best_centers = model.cluster_centers_

print("\nBest k-means cost:", best_cost)

### Which animals belong to each cluster in the best k-means solution.

In [ ]:
for i in range(10):
    members = [animals[j] for j in range(len(animals))
               if best_labels[j] == i]

    print("Cluster", i + 1, ":", members)